# Main Pipeline - Tong hop cong viec nhom (Giai doan 1 va 2)

Notebook nay chi tong hop lai cac phan da duoc cac thanh vien thuc hien trong giai doan 1-2:
- Data ingestion (tu module data_loader)
- EDA (tu module eda)
- Preprocessing: imputation -> encoding -> scaling (tu module preprocessing)

Luu y:
- Stage 3-4 duoc danh dau placeholder de giu dung pham vi bao cao tong hop.
- Notebook khong tu phat trien them logic ngoai phan da co tu code nhom.

## 1. Setup va Imports

In [14]:
from pathlib import Path
import os
import sys
import warnings

import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"Project root: {ROOT}")
print("Setup completed for Stage 1-2 aggregation")

Project root: c:\Users\Thanh Dung\Documents\MYDATA\BKU\4\2\ML\BTL\project_folder
Setup completed for Stage 1-2 aggregation


## 2. Load Config va Modules

In [15]:
from config import CONFIG
from modules.data_loader import load_data_from_config, create_dataframe
from modules.eda import generate_eda_report, plot_missing_values
from modules.preprocessing import preprocess_pipeline

for folder in ["reports/eda", "reports/evaluation", "features", "data/processed"]:
    (ROOT / folder).mkdir(parents=True, exist_ok=True)

print("CONFIG loaded")
print("Target:", CONFIG["data"]["target_column"])

CONFIG loaded
Target: is_canceled


## 3. Pipeline Orchestration

### 3.1 Data Ingestion

In [16]:
data_cfg = CONFIG["data"]
target_col = data_cfg["target_column"]
local_file = ROOT / data_cfg["file_path"]

if local_file.exists():
    print(f"[Data] Use local cache: {local_file}")
    df, metadata = create_dataframe(str(local_file))
else:
    print("[Data] Local cache not found. Download from configured source...")
    df, metadata = load_data_from_config()

print(f"Data shape: {df.shape}")
print(f"Rows: {metadata['n_rows']} | Columns: {metadata['n_columns']} | Missing: {metadata['total_missing']}")
display(df.head())

[Data] Use local cache: c:\Users\Thanh Dung\Documents\MYDATA\BKU\4\2\ML\BTL\project_folder\data\hotel_bookings.csv
[Data Loader] Đọc dữ liệu thành công - shape: (119390, 32)
Data shape: (119390, 32)
Rows: 119390 | Columns: 32 | Missing: 129425


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [17]:
# 3.2 EDA
eda_output_dir = ROOT / CONFIG["eda"]["output_dir"]
eda_output_dir.mkdir(parents=True, exist_ok=True)

eda_report = generate_eda_report(df, output_dir=str(eda_output_dir))
missing_plot_path = eda_output_dir / "missing_values_chart.png"
_ = plot_missing_values(df, output_path=str(missing_plot_path))

print("EDA completed")
print("EDA report:", eda_output_dir / "eda_report.json")
print("Missing chart:", missing_plot_path)

missing_focus = eda_report["missing_values_analysis"]["target_columns_missing"]
print("Top missing columns:")
for col, val in sorted(missing_focus.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"- {col}: {val}")

[EDA] Tạo báo cáo thống kê mô tả
[EDA] Đã lưu báo cáo thống kê mô tả tại: c:\Users\Thanh Dung\Documents\MYDATA\BKU\4\2\ML\BTL\project_folder\reports\eda\eda_report.json
[EDA] Vẽ biểu đồ missing values và lưu vào c:\Users\Thanh Dung\Documents\MYDATA\BKU\4\2\ML\BTL\project_folder\reports\eda\missing_values_chart.png
[EDA] Đang vẽ biểu đồ missing values và lưu vào c:\Users\Thanh Dung\Documents\MYDATA\BKU\4\2\ML\BTL\project_folder\reports\eda\missing_values_chart.png...
[EDA] Hoàn tất lưu biểu đồ tại: c:\Users\Thanh Dung\Documents\MYDATA\BKU\4\2\ML\BTL\project_folder\reports\eda\missing_values_chart.png
EDA completed
EDA report: c:\Users\Thanh Dung\Documents\MYDATA\BKU\4\2\ML\BTL\project_folder\reports\eda\eda_report.json
Missing chart: c:\Users\Thanh Dung\Documents\MYDATA\BKU\4\2\ML\BTL\project_folder\reports\eda\missing_values_chart.png
Top missing columns:
- company: 112593
- agent: 16340
- country: 488
- children: 4


In [18]:
# 3.3 Preprocessing
X, y, preprocessing_metadata = preprocess_pipeline(
    df=df,
    target_column=target_col,
    config=CONFIG["preprocessing"],
)

X = np.asarray(X)
y = np.asarray(y)

print("Preprocessing completed")
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Imputation:", preprocessing_metadata["imputation"]["method"])
print("Encoding:", preprocessing_metadata["encoding"]["method"])
print("Scaling:", preprocessing_metadata["scaling"]["method"])

[Preprocessing] Bắt đầu chạy full preprocessing pipeline...

--- BƯỚC 1: XỬ LÝ MISSING DATA (IMPUTATION) ---
[Preprocessing] Đã xoá các cột do missing values quá cao: ['company']
[Preprocessing] Đã điền giá trị 0.0 cho missing values của cột 'agent'
[Preprocessing] Áp dụng SimpleImputer với strategy='median' cho numeric features
[Preprocessing] Đã xử lý xong. Số cột còn missing data: 0
[Preprocessing] Đã lưu dataset sau khi xử lý missing data tại: data/processed\dataset_after_imputation.csv

--- BƯỚC 2: ENCODING ---
[Preprocessing] Áp dụng OneHot encoding cho 12 categorical columns: ['hotel', 'arrival_date_month', 'meal', 'country', 'market_segment', 'distribution_channel', 'reserved_room_type', 'assigned_room_type', 'deposit_type', 'customer_type', 'reservation_status', 'reservation_status_date']
[Preprocessing] OneHot encoding hoàn thành. Số features: 30 → 1185

--- BƯỚC 3: SCALING ---
[Preprocessing] Áp dụng StandardScaler cho 1185 numeric features
  - Mean range: [0.0000, 2016.1566

### 3.4 Stage 3 Placeholder (Feature Engineering)

Khong trien khai trong notebook tong hop giai doan 1-2.

In [19]:
# Placeholder only - Stage 3 khong nam trong pham vi tong hop giai doan 1-2.
print("[Placeholder] Stage 3 Feature Engineering chua tong hop trong notebook nay.")

[Placeholder] Stage 3 Feature Engineering chua tong hop trong notebook nay.


### 3.4 Stage 3 Placeholder (Feature Engineering)

Phan nay chua duoc tong hop trong pham vi giai doan 1-2. Khong thuc thi tai notebook nay.

In [20]:
# Placeholder only - Stage 4 model training khong nam trong pham vi tong hop giai doan 1-2.
print("[Placeholder] Stage 4 Model Training chua tong hop trong notebook nay.")

[Placeholder] Stage 4 Model Training chua tong hop trong notebook nay.


### 3.5-3.6 Stage 4 Placeholder (Model Training va Evaluation)

Phan nay chua duoc tong hop trong pham vi giai doan 1-2. Khong thuc thi tai notebook nay.

In [21]:
# Placeholder only - Stage 4 evaluation khong nam trong pham vi tong hop giai doan 1-2.
print("[Placeholder] Stage 4 Evaluation chua tong hop trong notebook nay.")

[Placeholder] Stage 4 Evaluation chua tong hop trong notebook nay.


## 4. Tom tat tong hop cong viec thanh vien

Notebook nay chi tong hop lai cac phan da duoc nhom hoan thanh trong giai doan 1-2:
- Data ingestion tu code module data_loader
- EDA tu code module eda
- Preprocessing tu code module preprocessing

Cac phan Stage 3-4 duoc de o dang placeholder de bao toan dung pham vi nhiem vu tong hop.

Artifacts duoc tao tu Stage 1-2:
- reports/eda/eda_report.json
- reports/eda/missing_values_chart.png
- data/processed/dataset_after_imputation.csv